In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

ROUNDS_PATH = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv")
OUT_DIR     = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- modeling knobs (keep what worked, tune later) ---
WINDOW_ROUNDS = 40
MIN_PRIOR_ROUNDS = 20
MIN_EVENT_ROUNDS = 2
MIN_COURSE_PLAYER_EVENTS = 200
SHRINK_K = 800

# --- outputs (OVERWRITE) ---
ABS_PARQUET = OUT_DIR / "course_fit_weights_predictive_2017_2026.parquet"
ABS_CSV     = OUT_DIR / "course_fit_weights_predictive_2017_2026.csv"
REL_PARQUET = OUT_DIR / "course_fit_weights_predictive_2017_2026_relative.parquet"
REL_CSV     = OUT_DIR / "course_fit_weights_predictive_2017_2026_relative.csv"

# ----------------------------
# Load minimal columns
# ----------------------------
needed = {
    "tour","year","event_id","event_name","course_num","course_name",
    "dg_id","player_name","round_num","round_date",
    "sg_total","sg_ott","sg_app","sg_arg","sg_putt",
}
df = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in needed, low_memory=False)

# numeric cleanup
num_cols = ["year","event_id","course_num","dg_id","round_num","sg_total","sg_ott","sg_app","sg_arg","sg_putt"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["tour","year","event_id","course_num","dg_id","round_num","sg_total"]).copy()
df["tour"] = df["tour"].astype(str)
df["year"] = df["year"].astype(int)
df["event_id"] = df["event_id"].astype(int)
df["course_num"] = df["course_num"].astype(int)
df["dg_id"] = df["dg_id"].astype(int)
df["round_num"] = df["round_num"].astype(int)

# skills required for predictors (history must have split SG)
skill_cols = ["sg_ott","sg_app","sg_arg","sg_putt"]
for c in skill_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Sort to create a stable per-player-tour timeline (no dates required)
df = df.sort_values(["tour","dg_id","year","event_id","round_num"]).copy()
df["pt_round_idx"] = df.groupby(["tour","dg_id"]).cumcount()

# First index of each (tour,year,event,player) for lookback cutoff
event_first_idx = (
    df.groupby(["tour","year","event_id","dg_id"], as_index=False)["pt_round_idx"]
      .min()
      .rename(columns={"pt_round_idx":"event_first_pt_idx"})
)

# Event outcome table (player-event) with event_total and number of rounds
event_outcome = (
    df.groupby(["tour","year","event_id","course_num","dg_id"], as_index=False)
      .agg(event_sg_total=("sg_total","sum"),
           event_rounds=("sg_total","size"))
      .merge(event_first_idx, on=["tour","year","event_id","dg_id"], how="left")
)

# Names + course names for convenience
names = df[["tour","dg_id","player_name"]].drop_duplicates()
course_name_map = (
    df.dropna(subset=["course_name"])
      .groupby("course_num")["course_name"]
      .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
      .to_dict()
)

# ----------------------------
# Fit one course
# ----------------------------
def fit_one_course(course_num: int) -> dict:
    eo = event_outcome[event_outcome["course_num"] == int(course_num)].copy()
    if eo.empty:
        return {"course_num": int(course_num), "status": "no rows"}

    rows = []
    for r in eo.itertuples(index=False):
        tour = r.tour
        year = int(r.year)
        event_id = int(r.event_id)
        dg_id = int(r.dg_id)
        y = float(r.event_sg_total)
        erounds = int(r.event_rounds)
        first_idx = r.event_first_pt_idx

        if pd.isna(first_idx):
            continue
        if erounds < MIN_EVENT_ROUNDS:
            continue

        # history strictly before event starts, same tour
        hist = df[
            (df["tour"] == tour) &
            (df["dg_id"] == dg_id) &
            (df["pt_round_idx"] < first_idx)
        ].tail(WINDOW_ROUNDS)

        # require enough prior rounds
        if hist["sg_total"].notna().sum() < MIN_PRIOR_ROUNDS:
            continue

        # require split-SG availability in history (predictors)
        feats = {}
        ok = True
        for c in skill_cols:
            v = hist[c].mean()
            if pd.isna(v):
                ok = False
                break
            feats[f"{c}_pre"] = float(v)
        if not ok:
            continue

        feats.update({
            "tour": tour,
            "year": year,
            "event_id": event_id,
            "course_num": int(course_num),
            "dg_id": dg_id,
            "event_sg_total": y,
            "event_rounds": erounds,
            "prior_rounds_used": int(hist.shape[0]),
        })
        rows.append(feats)

    pe = pd.DataFrame(rows)
    n = int(len(pe))
    if n < MIN_COURSE_PLAYER_EVENTS:
        return {
            "course_num": int(course_num),
            "course_name": course_name_map.get(int(course_num)),
            "status": f"insufficient ({n})",
            "n_player_events": n,
        }

    pe = pe.merge(names, on=["tour","dg_id"], how="left")

    need_pre = [f"{c}_pre" for c in skill_cols]
    pe = pe.dropna(subset=need_pre + ["event_sg_total"]).copy()

    # standardize within this course sample (pre-event skills)
    mu = pe[need_pre].mean()
    sd = pe[need_pre].std(ddof=0).replace(0, np.nan)

    for c in need_pre:
        pe[f"z_{c}"] = (pe[c] - mu[c]) / sd[c]

    zcols = [f"z_{c}" for c in need_pre]
    pe = pe.dropna(subset=zcols).copy()

    X = sm.add_constant(pe[zcols])
    y = pe["event_sg_total"]
    m = sm.OLS(y, X).fit()

    beta_ott  = float(m.params.get("z_sg_ott_pre", np.nan))
    beta_app  = float(m.params.get("z_sg_app_pre", np.nan))
    beta_arg  = float(m.params.get("z_sg_arg_pre", np.nan))
    beta_putt = float(m.params.get("z_sg_putt_pre", np.nan))

    # shrink toward 0
    w = n / (n + SHRINK_K)
    beta_ott_s  = w * beta_ott
    beta_app_s  = w * beta_app
    beta_arg_s  = w * beta_arg
    beta_putt_s = w * beta_putt

    pred = float(np.sqrt(beta_ott_s**2 + beta_app_s**2 + beta_arg_s**2 + beta_putt_s**2))
    abs_sum = abs(beta_ott_s) + abs(beta_app_s) + abs(beta_arg_s) + abs(beta_putt_s)

    if abs_sum == 0:
        imp_ott = imp_app = imp_arg = imp_putt = np.nan
    else:
        imp_ott  = abs(beta_ott_s)  / abs_sum
        imp_app  = abs(beta_app_s)  / abs_sum
        imp_arg  = abs(beta_arg_s)  / abs_sum
        imp_putt = abs(beta_putt_s) / abs_sum

    return {
        "course_num": int(course_num),
        "course_name": course_name_map.get(int(course_num)),
        "status": "ok",
        "n_player_events": n,
        "r2": float(m.rsquared),
        "randomness": float(1.0 - m.rsquared),  # higher = more random (your requested identifier)
        "beta_ott": beta_ott,
        "beta_app": beta_app,
        "beta_arg": beta_arg,
        "beta_putt": beta_putt,
        "beta_ott_shrunk": beta_ott_s,
        "beta_app_shrunk": beta_app_s,
        "beta_arg_shrunk": beta_arg_s,
        "beta_putt_shrunk": beta_putt_s,
        "predictability": pred,
        "imp_ott": imp_ott,
        "imp_app": imp_app,
        "imp_arg": imp_arg,
        "imp_putt": imp_putt,
    }

# ----------------------------
# Run ALL courses (that appear in event_outcome)
# ----------------------------
all_courses = sorted(event_outcome["course_num"].dropna().unique().tolist())
print(f"Courses to evaluate: {len(all_courses)} (will keep those with >= {MIN_COURSE_PLAYER_EVENTS} player-events)")

results = []
for c in all_courses:
    results.append(fit_one_course(int(c)))

abs_df = pd.DataFrame(results)

# keep only ok for final absolute weights file
abs_ok = abs_df[abs_df["status"] == "ok"].copy().reset_index(drop=True)

# overwrite absolute file
abs_ok.to_parquet(ABS_PARQUET, index=False)
abs_ok.to_csv(ABS_CSV, index=False)

print(f"\nOVERWROTE absolute course weights:\n- {ABS_PARQUET}\n- {ABS_CSV}")
print(f"Kept {len(abs_ok)} courses with status=ok.\n")
display(abs_ok.sort_values("predictability", ascending=False).head(20))

# ----------------------------
# Build RELATIVE-to-other-courses file
# - z-scores and percentiles for shrunk betas + predictability/randomness
# - plus DataGolf-like 0..1 scaling across courses per attribute (optional)
# ----------------------------
rel = abs_ok.copy()

rel_cols = ["beta_ott_shrunk","beta_app_shrunk","beta_arg_shrunk","beta_putt_shrunk","predictability","randomness","r2"]
for col in rel_cols:
    mu = rel[col].mean()
    sd = rel[col].std(ddof=0)
    rel[f"{col}_z"] = (rel[col] - mu) / sd if sd and sd > 0 else np.nan
    rel[f"{col}_pct"] = rel[col].rank(pct=True)

# 0..1 scaling across courses per attribute (useful for a radar view comparing courses)
for col in ["beta_ott_shrunk","beta_app_shrunk","beta_arg_shrunk","beta_putt_shrunk"]:
    cmin = rel[col].min()
    cmax = rel[col].max()
    rel[f"{col}_minmax01"] = (rel[col] - cmin) / (cmax - cmin) if cmax > cmin else np.nan

# also keep within-course shares (imp_*) already in rel

# overwrite relative file
rel.to_parquet(REL_PARQUET, index=False)
rel.to_csv(REL_CSV, index=False)

print(f"\nOVERWROTE relative course weights:\n- {REL_PARQUET}\n- {REL_CSV}")
display(rel.sort_values("randomness", ascending=False).head(20))


Courses to evaluate: 500 (will keep those with >= 200 player-events)

OVERWROTE absolute course weights:
- /Users/joshmacbook/python_projects/OAD/Data/MST/course_fit_weights_predictive_2017_2026.parquet
- /Users/joshmacbook/python_projects/OAD/Data/MST/course_fit_weights_predictive_2017_2026.csv
Kept 60 courses with status=ok.



,course_num,course_name,status,n_player_events,r2,randomness,beta_ott,beta_app,beta_arg,beta_putt,beta_ott_shrunk,beta_app_shrunk,beta_arg_shrunk,beta_putt_shrunk,predictability,imp_ott,imp_app,imp_arg,imp_putt
17,503,TPC River Highlands,ok,1174,0.158214,0.841786,1.339952,1.051558,0.637356,0.569231,0.796912,0.625395,0.379056,0.338540,1.133348,0.372406,0.292254,0.177137,0.158203
15,244,Corales Golf Club,ok,900,0.157539,0.842461,1.477347,0.820184,0.776120,0.620307,0.782125,0.434215,0.410887,0.328398,1.037755,0.399936,0.222034,0.210105,0.167925
43,883,TPC Twin Cities,ok,1013,0.136413,0.863587,1.138529,0.946464,0.778871,0.778258,0.636144,0.528830,0.435188,0.434846,1.030932,0.312600,0.259866,0.213851,0.213683
4,11,TPC Sawgrass,ok,1039,0.107476,0.892524,1.323310,0.765984,0.659373,0.678144,0.747645,0.432766,0.372533,0.383138,1.015793,0.386164,0.223527,0.192416,0.197894
24,669,TPC Deere Run,ok,1074,0.119045,0.880955,1.158878,1.009017,0.637103,0.551786,0.664159,0.578273,0.365127,0.316232,1.004404,0.345235,0.300590,0.189796,0.164379
8,23,Muirfield Village Golf Club,ok,927,0.104395,0.895605,1.215584,0.774543,0.774475,0.766152,0.652488,0.415751,0.415714,0.411247,0.969810,0.344284,0.219370,0.219351,0.216994
44,884,Keene Trace Golf Club,ok,543,0.172524,0.827476,1.955106,0.731980,0.893014,0.622631,0.790486,0.295953,0.361062,0.251741,0.951943,0.465199,0.174168,0.212484,0.148149
6,14,Augusta National Golf Club,ok,660,0.154628,0.845372,1.333699,0.786382,1.345667,0.463632,0.602905,0.355488,0.608315,0.209587,0.950705,0.339417,0.200129,0.342463,0.117991
18,510,TPC Scottsdale,ok,1117,0.096035,0.903965,1.218109,0.615813,0.666555,0.548882,0.709769,0.358822,0.388389,0.319824,0.941095,0.399464,0.201948,0.218589,0.179999
5,12,Harbour Town Golf Links,ok,995,0.113818,0.886182,1.207037,0.673036,0.744152,0.643011,0.669082,0.373075,0.412497,0.356432,0.940241,0.369437,0.205995,0.227762,0.196806



OVERWROTE relative course weights:
- /Users/joshmacbook/python_projects/OAD/Data/MST/course_fit_weights_predictive_2017_2026_relative.parquet
- /Users/joshmacbook/python_projects/OAD/Data/MST/course_fit_weights_predictive_2017_2026_relative.csv


,course_num,course_name,status,n_player_events,r2,randomness,beta_ott,beta_app,beta_arg,beta_putt,...,predictability_z,predictability_pct,randomness_z,randomness_pct,r2_z,r2_pct,beta_ott_shrunk_minmax01,beta_app_shrunk_minmax01,beta_arg_shrunk_minmax01,beta_putt_shrunk_minmax01
1,5,Pebble Beach Golf Links,ok,629,0.015773,0.984227,0.247969,0.359102,0.215880,0.241388,...,-1.822901,0.016667,2.510960,1.000000,-2.510960,0.016667,0.062693,0.133032,0.171146,0.364284
27,704,Stadium Course,ok,601,0.028366,0.971634,0.507398,0.359537,0.075300,-0.076576,...,-1.687603,0.033333,2.182481,0.983333,-2.182481,0.033333,0.210582,0.125924,0.069865,0.155934
41,875,Accordia Golf Narashino Country Club,ok,339,0.055769,0.944231,1.050008,0.544414,0.252862,0.167911,...,-1.289605,0.083333,1.467696,0.966667,-1.467696,0.050000,0.339846,0.140394,0.139231,0.279992
19,513,TPC Southwind,ok,752,0.056346,0.943654,0.793708,0.529639,0.617347,0.250678,...,-0.428989,0.433333,1.452651,0.950000,-1.452651,0.066667,0.438061,0.315883,0.500728,0.387068
55,928,PGA National Resort (The Champion Course),ok,255,0.063841,0.936159,0.662419,0.883825,0.470080,0.127991,...,-1.597165,0.050000,1.257147,0.933333,-1.257147,0.083333,0.132147,0.236105,0.201178,0.251474
34,752,Sedgefield Country Club,ok,1202,0.064875,0.935125,0.429518,1.020051,0.442707,0.415445,...,0.403905,0.616667,1.230183,0.916667,-1.230183,0.100000,0.265393,0.975964,0.446915,0.578747
36,770,TPC San Antonio (Oaks Course),ok,1107,0.070911,0.929089,0.805239,0.637185,0.767947,0.371708,...,0.477117,0.666667,1.072721,0.900000,-1.072721,0.116667,0.550980,0.525984,0.737554,0.528332
38,776,Sea Island GC (Seaside),ok,586,0.071726,0.928274,0.572738,0.705781,0.466338,0.727193,...,-0.573782,0.366667,1.051461,0.883333,-1.051461,0.133333,0.243957,0.393382,0.336087,0.665658
0,4,Torrey Pines GC (South),ok,642,0.073948,0.926052,0.870843,0.719296,0.262251,0.462120,...,-0.464081,0.400000,0.993502,0.866667,-0.993502,0.150000,0.442330,0.433893,0.206243,0.513306
46,886,Liberty National Golf Club,ok,237,0.073968,0.926032,0.850685,0.850938,0.651045,0.178332,...,-1.496176,0.066667,0.992992,0.850000,-0.992992,0.166667,0.178903,0.200580,0.257971,0.266184


In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

ROUNDS_PATH = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv")
OUT_DIR     = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use")

# ---- knobs ----
WINDOW_ROUNDS = 40
MIN_PRIOR_ROUNDS = 20
MIN_EVENT_ROUNDS = 2
MIN_COURSE_PLAYER_EVENTS = 200
SHRINK_K = 800  # shrink betas toward global mean (computed across courses after fit)

# outputs (OVERWRITE)
ABS_PARQUET = OUT_DIR / "course_fit_weights_predictive_2017_2026.parquet"
ABS_CSV     = OUT_DIR / "course_fit_weights_predictive_2017_2026.csv"
REL_PARQUET = OUT_DIR / "course_fit_weights_predictive_2017_2026_relative.parquet"
REL_CSV     = OUT_DIR / "course_fit_weights_predictive_2017_2026_relative.csv"

# ----------------------------
# Load minimal columns
# ----------------------------
needed = {
    "tour","year","event_id","event_name","course_num","course_name",
    "dg_id","player_name","round_num","round_date",
    "sg_total","sg_ott","sg_app","sg_arg","sg_putt",
}
df = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in needed, low_memory=False)

num_cols = ["year","event_id","course_num","dg_id","round_num","sg_total","sg_ott","sg_app","sg_arg","sg_putt"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["tour","year","event_id","course_num","dg_id","round_num","sg_total"]).copy()
df["tour"] = df["tour"].astype(str)
df["year"] = df["year"].astype(int)
df["event_id"] = df["event_id"].astype(int)
df["course_num"] = df["course_num"].astype(int)
df["dg_id"] = df["dg_id"].astype(int)
df["round_num"] = df["round_num"].astype(int)

skill_cols = ["sg_ott","sg_app","sg_arg","sg_putt"]
for c in skill_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# stable per-player-tour timeline (no dates required)
df = df.sort_values(["tour","dg_id","year","event_id","round_num"]).copy()
df["pt_round_idx"] = df.groupby(["tour","dg_id"]).cumcount()

# first round index per player-event
event_first_idx = (
    df.groupby(["tour","year","event_id","dg_id"], as_index=False)["pt_round_idx"]
      .min()
      .rename(columns={"pt_round_idx":"event_first_pt_idx"})
)

# player-event outcome
event_outcome = (
    df.groupby(["tour","year","event_id","course_num","dg_id"], as_index=False)
      .agg(event_sg_total=("sg_total","sum"),
           event_rounds=("sg_total","size"))
      .merge(event_first_idx, on=["tour","year","event_id","dg_id"], how="left")
)

# names/maps
names = df[["tour","dg_id","player_name"]].drop_duplicates()
course_name_map = (
    df.dropna(subset=["course_name"])
      .groupby("course_num")["course_name"]
      .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
      .to_dict()
)

# ----------------------------
# Build GLOBAL player-event dataset (pe)
# predictors: pre-event rolling means (same-tour history)
# ----------------------------
rows = []
eo = event_outcome.copy()

for r in eo.itertuples(index=False):
    tour = r.tour
    year = int(r.year)
    event_id = int(r.event_id)
    course_num = int(r.course_num)
    dg_id = int(r.dg_id)
    y = float(r.event_sg_total)
    erounds = int(r.event_rounds)
    first_idx = r.event_first_pt_idx

    if pd.isna(first_idx):
        continue
    if erounds < MIN_EVENT_ROUNDS:
        continue

    hist = df[
        (df["tour"] == tour) &
        (df["dg_id"] == dg_id) &
        (df["pt_round_idx"] < first_idx)
    ].tail(WINDOW_ROUNDS)

    if hist["sg_total"].notna().sum() < MIN_PRIOR_ROUNDS:
        continue

    # require split SG in history
    feats = {}
    ok = True
    for c in skill_cols:
        v = hist[c].mean()
        if pd.isna(v):
            ok = False
            break
        feats[f"{c}_pre"] = float(v)
    if not ok:
        continue

    feats.update({
        "tour": tour,
        "year": year,
        "event_id": event_id,
        "course_num": course_num,
        "dg_id": dg_id,
        "event_sg_total": y,
        "event_rounds": erounds,
        "prior_rounds_used": int(hist.shape[0]),
    })
    rows.append(feats)

pe = pd.DataFrame(rows).merge(names, on=["tour","dg_id"], how="left")
print(f"Global player-event rows built: {len(pe):,}")

# ----------------------------
# GLOBAL z-scores for pre-event skills (this is the key change)
# ----------------------------
pre_cols = [f"{c}_pre" for c in skill_cols]
mu = pe[pre_cols].mean()
sd = pe[pre_cols].std(ddof=0).replace(0, np.nan)

for c in pre_cols:
    pe[f"z_{c}"] = (pe[c] - mu[c]) / sd[c]

zcols = [f"z_{c}" for c in pre_cols]
pe = pe.dropna(subset=zcols + ["event_sg_total"]).copy()

# ----------------------------
# Fit per-course regressions using GLOBAL zcols
# ----------------------------
def fit_course(course_num: int, g: pd.DataFrame) -> dict:
    n = int(len(g))
    if n < MIN_COURSE_PLAYER_EVENTS:
        return {
            "course_num": int(course_num),
            "course_name": course_name_map.get(int(course_num)),
            "status": f"insufficient ({n})",
            "n_player_events": n,
        }

    X = sm.add_constant(g[zcols])
    y = g["event_sg_total"]
    m = sm.OLS(y, X).fit()

    beta_ott  = float(m.params.get("z_sg_ott_pre", np.nan))
    beta_app  = float(m.params.get("z_sg_app_pre", np.nan))
    beta_arg  = float(m.params.get("z_sg_arg_pre", np.nan))
    beta_putt = float(m.params.get("z_sg_putt_pre", np.nan))

    return {
        "course_num": int(course_num),
        "course_name": course_name_map.get(int(course_num)),
        "status": "ok",
        "n_player_events": n,
        "r2": float(m.rsquared),
        "randomness": float(1.0 - m.rsquared),
        "beta_ott": beta_ott,
        "beta_app": beta_app,
        "beta_arg": beta_arg,
        "beta_putt": beta_putt,
    }

course_rows = []
for cnum, g in pe.groupby("course_num", sort=False):
    course_rows.append(fit_course(int(cnum), g))

abs_df = pd.DataFrame(course_rows)
abs_ok = abs_df[abs_df["status"] == "ok"].copy().reset_index(drop=True)
print(f"Courses kept (status=ok): {len(abs_ok)}")

# ----------------------------
# Shrink betas toward GLOBAL mean across courses (DataGolf-ish stabilization)
# ----------------------------
beta_cols = ["beta_ott","beta_app","beta_arg","beta_putt"]
global_beta_means = abs_ok[beta_cols].mean()

w = abs_ok["n_player_events"] / (abs_ok["n_player_events"] + SHRINK_K)
for b in beta_cols:
    abs_ok[f"{b}_shrunk"] = w * abs_ok[b] + (1 - w) * float(global_beta_means[b])

abs_ok["predictability"] = np.sqrt(
    abs_ok["beta_ott_shrunk"]**2
    + abs_ok["beta_app_shrunk"]**2
    + abs_ok["beta_arg_shrunk"]**2
    + abs_ok["beta_putt_shrunk"]**2
)

abs_sum = (
    abs_ok[["beta_ott_shrunk","beta_app_shrunk","beta_arg_shrunk","beta_putt_shrunk"]]
    .abs()
    .sum(axis=1)
    .replace(0, np.nan)
)
abs_ok["imp_ott"]  = abs_ok["beta_ott_shrunk"].abs()  / abs_sum
abs_ok["imp_app"]  = abs_ok["beta_app_shrunk"].abs()  / abs_sum
abs_ok["imp_arg"]  = abs_ok["beta_arg_shrunk"].abs()  / abs_sum
abs_ok["imp_putt"] = abs_ok["beta_putt_shrunk"].abs() / abs_sum

# primary skill label
def _primary_skill(row):
    d = {
        "OTT": abs(row["beta_ott_shrunk"]),
        "APP": abs(row["beta_app_shrunk"]),
        "ARG": abs(row["beta_arg_shrunk"]),
        "PUTT": abs(row["beta_putt_shrunk"]),
    }
    return max(d, key=d.get)

abs_ok["primary_skill"] = abs_ok.apply(_primary_skill, axis=1)

# percentiles for easy UI labels
abs_ok["predictability_pct"] = abs_ok["predictability"].rank(pct=True)
abs_ok["randomness_pct"]     = abs_ok["randomness"].rank(pct=True)
abs_ok["r2_pct"]             = abs_ok["r2"].rank(pct=True)

# save ABS (overwrite)
abs_ok.to_parquet(ABS_PARQUET, index=False)
abs_ok.to_csv(ABS_CSV, index=False)
print(f"\nOVERWROTE ABS:\n- {ABS_PARQUET}\n- {ABS_CSV}")
display(abs_ok.sort_values("predictability", ascending=False).head(20))

# ----------------------------
# Build RELATIVE file (course vs all courses)
# - z-scores + percentiles for shrunk betas and predictability/randomness
# ----------------------------
rel = abs_ok.copy()

rel_base = ["beta_ott_shrunk","beta_app_shrunk","beta_arg_shrunk","beta_putt_shrunk","predictability","randomness","r2"]
for col in rel_base:
    m = rel[col].mean()
    s = rel[col].std(ddof=0)
    rel[f"{col}_z"] = (rel[col] - m) / s if s and s > 0 else np.nan
    rel[f"{col}_pct"] = rel[col].rank(pct=True)

# save REL (overwrite)
rel.to_parquet(REL_PARQUET, index=False)
rel.to_csv(REL_CSV, index=False)
print(f"\nOVERWROTE REL:\n- {REL_PARQUET}\n- {REL_CSV}")
display(rel.sort_values("randomness", ascending=False).head(20))


Global player-event rows built: 46,014
Courses kept (status=ok): 60

OVERWROTE ABS:
- /Users/joshmacbook/python_projects/OAD/Data/MST/course_fit_weights_predictive_2017_2026.parquet
- /Users/joshmacbook/python_projects/OAD/Data/MST/course_fit_weights_predictive_2017_2026.csv


,course_num,course_name,status,n_player_events,r2,randomness,beta_ott,beta_app,beta_arg,beta_putt,...,beta_putt_shrunk,predictability,imp_ott,imp_app,imp_arg,imp_putt,primary_skill,predictability_pct,randomness_pct,r2_pct
7,656,Plantation Course at Kapalua,ok,385,0.192011,0.807989,1.902605,1.235522,1.053564,0.276619,...,0.470766,1.924897,0.392745,0.267659,0.208673,0.130923,OTT,1.000000,0.016667,1.000000
15,503,TPC River Highlands,ok,1219,0.158214,0.841786,1.482198,1.091123,0.659018,0.617721,...,0.596514,1.894868,0.379846,0.275770,0.177895,0.166489,OTT,0.983333,0.133333,0.883333
6,14,Augusta National Golf Club,ok,689,0.154628,0.845372,1.354012,0.836306,1.393792,0.484417,...,0.527282,1.869953,0.350659,0.232233,0.270188,0.146920,OTT,0.966667,0.183333,0.833333
36,872,Quail Hollow Club,ok,679,0.151294,0.848706,1.562023,0.992197,0.770564,0.587553,...,0.574921,1.855914,0.385131,0.257547,0.193710,0.163611,OTT,0.950000,0.216667,0.800000
41,884,Keene Trace Golf Club,ok,567,0.172524,0.827476,1.745551,0.697716,0.827654,0.576277,...,0.569209,1.845618,0.408870,0.224567,0.201779,0.164783,OTT,0.933333,0.050000,0.966667
3,11,TPC Sawgrass,ok,1094,0.107476,0.892524,1.494134,0.851997,0.695816,0.693500,...,0.638885,1.843947,0.388631,0.240959,0.187823,0.182587,OTT,0.916667,0.516667,0.500000
23,240,Club de Golf Chapultepec,ok,247,0.156122,0.843878,2.008553,0.465392,1.099730,0.963951,...,0.658506,1.841857,0.392488,0.212964,0.206246,0.188301,OTT,0.900000,0.166667,0.850000
39,874,Hamilton Golf & Country Club,ok,280,0.154384,0.845616,1.566887,0.789619,1.623379,0.600650,...,0.573650,1.840748,0.360873,0.231716,0.245363,0.162048,OTT,0.883333,0.200000,0.816667
47,890,Tahoe Mt. Club (Old Greenwood),ok,471,0.171213,0.828787,1.637078,1.032967,0.324276,0.810256,...,0.655382,1.820734,0.395149,0.265737,0.146854,0.192261,OTT,0.866667,0.066667,0.950000
53,921,TPC Craig Ranch,ok,465,0.159789,0.840211,1.473138,1.136358,0.648641,0.580670,...,0.570254,1.803405,0.375887,0.275843,0.181507,0.166764,OTT,0.850000,0.116667,0.900000



OVERWROTE REL:
- /Users/joshmacbook/python_projects/OAD/Data/MST/course_fit_weights_predictive_2017_2026_relative.parquet
- /Users/joshmacbook/python_projects/OAD/Data/MST/course_fit_weights_predictive_2017_2026_relative.csv


,course_num,course_name,status,n_player_events,r2,randomness,beta_ott,beta_app,beta_arg,beta_putt,...,beta_ott_shrunk_pct,beta_app_shrunk_z,beta_app_shrunk_pct,beta_arg_shrunk_z,beta_arg_shrunk_pct,beta_putt_shrunk_z,beta_putt_shrunk_pct,predictability_z,randomness_z,r2_z
33,5,Pebble Beach Golf Links,ok,665,0.015773,0.984227,0.265124,0.380416,0.234243,0.251713,...,0.033333,-2.184803,0.016667,-1.369579,0.083333,-1.379496,0.083333,-3.431217,2.510960,-2.510960
30,704,Stadium Course,ok,641,0.028366,0.971634,0.566876,0.407732,0.080018,-0.076448,...,0.083333,-2.012650,0.033333,-1.912130,0.050000,-2.701285,0.016667,-3.128871,2.182481,-2.182481
44,875,Accordia Golf Narashino Country Club,ok,364,0.055769,0.944231,1.217755,0.542989,0.256245,0.172553,...,0.500000,-0.972050,0.183333,-0.877643,0.200000,-1.200605,0.100000,-0.772851,1.467696,-1.467696
11,513,TPC Southwind,ok,792,0.056346,0.943654,0.842205,0.566940,0.669387,0.266167,...,0.183333,-1.409643,0.066667,0.296677,0.616667,-1.438802,0.066667,-1.488408,1.452651,-1.452651
56,928,PGA National Resort (The Champion Course),ok,278,0.063841,0.936159,0.735712,0.958296,0.466208,0.143017,...,0.233333,0.329981,0.633333,-0.268571,0.400000,-1.072640,0.133333,-0.767294,1.257147,-1.257147
5,752,Sedgefield Country Club,ok,1261,0.064875,0.935125,0.467800,1.084471,0.431279,0.425562,...,0.016667,1.625226,0.933333,-0.853114,0.216667,-0.852924,0.216667,-1.691926,1.230183,-1.230183
16,770,TPC San Antonio (Oaks Course),ok,1162,0.070911,0.929089,0.854772,0.694683,0.797969,0.388731,...,0.166667,-0.873209,0.233333,0.981219,0.883333,-1.029286,0.150000,-1.084391,1.072721,-1.072721
27,776,Sea Island GC (Seaside),ok,613,0.071726,0.928274,0.626793,0.752729,0.470080,0.724702,...,0.116667,-0.377593,0.350000,-0.456819,0.316667,0.573408,0.733333,-1.195080,1.051461,-1.051461
32,4,Torrey Pines GC (South),ok,679,0.073948,0.926052,0.929440,0.787833,0.266369,0.451869,...,0.250000,-0.227801,0.433333,-1.262830,0.100000,-0.545838,0.250000,-1.109818,0.993502,-0.993502
38,886,Liberty National Golf Club,ok,245,0.073968,0.926032,1.117463,0.953138,0.736313,0.192365,...,0.400000,0.285564,0.616667,0.285020,0.600000,-0.874648,0.200000,-0.099177,0.992992,-0.992992
